# 实验10：数据湖仓构建者

## 从脏日志到可查询的 Parquet 数据湖

在真实数据工程中，数据很少是干净的。

你不会总是拿到一个完美的 CSV。更常见的情况是：

- 整数字段里混入了字符串
- 数值字段里出现了 `"zero"`
- 某些字段缺失
- 时间戳是字符串
- 上游应用写入了错误格式
- 下游分析师明天早上就要用这份数据做报表

本实验模拟一家流媒体公司的播放日志管道。

你的任务是把原始播放事件日志处理成可供分析的数据资产。

---

## 核心思想

> 初级工程师相信数据是干净的。  
> 高级工程师假设数据一定会坏。

---

## 本实验你将构建

```text
raw_logs.csv
    ↓
显式 Schema 读取
    ↓
Bronze 层：原始摄取数据
    ↓
Silver 层：清洗后的事件数据
    ↓
Gold 层：面向分析的聚合表
    ↓
按 device_type 分区的 Parquet 数据集
```

---

## 本实验必须做到

- 使用 `StructType` 显式定义 Schema
- 不使用 `inferSchema`
- 展示 `"CORRUPT"` 和 `"zero"` 如何变成 `null`
- 清洗数据并保留可用记录
- 使用 `explain()` 查看执行计划
- 写出分区 Parquet 文件
- 审计输出文件夹结构
- 最后调用 `spark.stop()`

# 1. 环境准备

现在不应该存在问题

# 2. 导入必要模块

本实验会用到：

- `SparkSession`：Spark 程序入口
- `functions as F`：Spark SQL 内置函数
- `StructType` / `StructField`：显式定义 Schema
- Python 的 `os` / `shutil`：创建文件与审计目录结构

In [ ]:
import os
import shutil

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)

# 3. 启动 SparkSession

`SparkSession` 是本实验所有 Spark 操作的入口。

在真实生产环境中，它可能连接到一个云端 Spark 集群。

In [ ]:
spark = SparkSession.builder \
    .appName("Lab10_DataLakeBuilder") \
    .getOrCreate()

spark

# 4. 检查 Spark 环境

这一步用于确认 Spark 已经正常启动。

In [ ]:
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Spark version: 4.1.1
Spark master: local[*]
Default parallelism: 32


# 5. 实验背景：流媒体播放日志

假设你在一家流媒体 company 工作。

移动端 App 会不断上传用户观看视频的事件日志。

每一行日志代表一次播放事件：

| 字段 | 含义 |
|---|---|
| `event_id` | 播放事件 ID |
| `user_id` | 用户 ID |
| `video_id` | 视频 ID |
| `duration_watched` | 观看时长，单位秒 |
| `device_type` | 设备类型 |
| `event_timestamp` | 事件发生时间 |

理想情况下，数据应该长这样：

```text
101,500,vid_a,120.5,iPhone,2023-01-01T10:00:00
```

但真实世界不会这么仁慈。

# 6. 创建一份脏 CSV 文件

我们现在创建一个模拟的原始日志文件：`raw_logs.csv`。

其中包含几种典型脏数据：

1. `user_id` 本应是整数，但出现了 `"CORRUPT"`
2. `duration_watched` 本应是数字，但出现了 `"zero"`
3. 某一行缺失了观看时长
4. 时间戳目前还是字符串

In [ ]:
data_content = """event_id,user_id,video_id,duration_watched,device_type,event_timestamp
101,500,vid_a,120.5,iPhone,2023-01-01T10:00:00
102,501,vid_b,30.0,Android,2023-01-01T10:05:00
103,CORRUPT,vid_c,zero,iPhone,2023-01-01T10:10:00
104,502,vid_a,,iPad,2023-01-01T10:15:00
105,503,vid_d,300.0,Android,2023-01-01T10:20:00
106,504,vid_e,15.5,iPhone,2023-01-01T10:25:00
"""

with open("raw_logs.csv", "w", encoding="utf-8") as f:
    f.write(data_content)

print("脏数据 raw_logs.csv 已创建。")

脏数据 raw_logs.csv 已创建。


# 7. 先用普通 Python 查看原始文件

注意：这一步不是 Spark 读取。

我们只是像数据工程师一样先看一眼上游到底给了什么。

In [ ]:
with open("raw_logs.csv", "r", encoding="utf-8") as f:
    print(f.read())

event_id,user_id,video_id,duration_watched,device_type,event_timestamp
101,500,vid_a,120.5,iPhone,2023-01-01T10:00:00
102,501,vid_b,30.0,Android,2023-01-01T10:05:00
103,CORRUPT,vid_c,zero,iPhone,2023-01-01T10:10:00
104,502,vid_a,,iPad,2023-01-01T10:15:00
105,503,vid_d,300.0,Android,2023-01-01T10:20:00
106,504,vid_e,15.5,iPhone,2023-01-01T10:25:00



## 观察问题

请你在原始文件中找出问题：

1. 哪一行的 `user_id` 有问题？
2. 哪一行的 `duration_watched` 有问题？
3. 哪一行的 `duration_watched` 缺失？
4. 如果我们不显式定义 Schema，Spark 有可能如何误判字段类型？

请先思考，再继续。

# 8. 为什么本实验禁止使用 `inferSchema`？

在很多入门教程里，你可能会看到：

```python
spark.read.csv(..., inferSchema=True)
```

这在小数据上看起来很方便。

但在真实数据工程中，我们通常不希望把生产管道建立在“猜测”之上。

---

## `inferSchema` 的问题

1. **需要额外扫描数据**

Spark 需要读取一部分或全部数据来猜类型。

2. **大数据上成本高**

如果数据是 TB 级别，这种扫描是有代价的。

3. **脏数据会干扰推断**

比如 `user_id` 里混入 `"CORRUPT"`，Spark 可能推断成字符串。

4. **生产管道不稳定**

今天推断为整数，明天因为一批坏数据变成字符串。

5. **Schema 应该是契约，不应该是猜测**

数据工程中的一个核心原则是：

> Schema is a contract.

也就是说，Schema 是上游和下游之间的契约。

# 9. 显式定义 Schema

我们现在明确告诉 Spark 每一列应该是什么类型。

注意：

- `event_id` 应该是整数
- `user_id` 应该是整数
- `video_id` 是字符串
- `duration_watched` 应该是浮点数
- `device_type` 是字符串
- `event_timestamp` 先作为字符串读取，后面再转换为 timestamp

---

## 必须掌握

本实验必须使用：

```python
StructType([
    StructField(...),
    ...
])
```

禁止使用：

```python
inferSchema=True
```

In [ ]:
schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("video_id", StringType(), True),
    StructField("duration_watched", DoubleType(), True),
    StructField("device_type", StringType(), True),
    StructField("event_timestamp", StringType(), True),
])

# 10. 解释 Schema 中的 `True`

每个 `StructField` 的第三个参数表示该列是否允许为空。

例如：

```python
StructField("user_id", IntegerType(), True)
```

意思是：

- 列名是 `user_id`
- 类型应该是整数
- 允许出现 `null`

为什么允许为空？

因为真实数据可能坏掉。我们允许 Spark 把无法解析的坏值先变成 `null`，然后再由我们制定清洗规则。

这比在读取时直接失败更适合本实验的教学目标。

# 11. Bronze 层：读取原始日志

在数据湖仓中，我们常用以下分层思想：

| 层级 | 含义 |
|---|---|
| Bronze | 原始摄取层，尽量保留上游数据 |
| Silver | 清洗标准化层，修复类型、空值、格式问题 |
| Gold | 面向业务分析或报表的高质量数据 |

现在我们读取 CSV，形成 Bronze DataFrame。

注意我们使用：

```python
.option("mode", "PERMISSIVE")
```

它的含义是：

> 遇到无法解析的字段时，尽量保留该行，并将无法解析的字段处理为 `null`。

这让我们可以先摄取，再制定清洗规则。

In [ ]:
df_bronze = spark.read \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .csv("raw_logs.csv")

# 12. 查看 Bronze 数据

现在观察 Spark 根据我们定义的 Schema 读取后的结果。

In [ ]:
df_bronze.show(truncate=False)

+--------+-------+--------+----------------+-----------+-------------------+
|event_id|user_id|video_id|duration_watched|device_type|event_timestamp    |
+--------+-------+--------+----------------+-----------+-------------------+
|101     |500    |vid_a   |120.5           |iPhone     |2023-01-01T10:00:00|
|102     |501    |vid_b   |30.0            |Android    |2023-01-01T10:05:00|
|103     |NULL   |vid_c   |NULL            |iPhone     |2023-01-01T10:10:00|
|104     |502    |vid_a   |NULL            |iPad       |2023-01-01T10:15:00|
|105     |503    |vid_d   |300.0           |Android    |2023-01-01T10:20:00|
|106     |504    |vid_e   |15.5            |iPhone     |2023-01-01T10:25:00|
+--------+-------+--------+----------------+-----------+-------------------+



# 13. 查看 Schema

请特别观察：

- `user_id` 是否是 integer
- `duration_watched` 是否是 double
- `event_timestamp` 目前是否仍然是 string

In [ ]:
df_bronze.printSchema()

root
 |-- event_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- video_id: string (nullable = true)
 |-- duration_watched: double (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_timestamp: string (nullable = true)



## 关键观察

原始 CSV 中有这一行：

```text
103,CORRUPT,vid_c,zero,iPhone,2023-01-01T10:10:00
```

但因为我们显式定义了：

```python
user_id: IntegerType
duration_watched: DoubleType
```

所以无法解析的：

- `"CORRUPT"` 会变成 `null`
- `"zero"` 会变成 `null`

这正是本实验要你证明的事情。

---

## 工程原则

> 不要让脏数据悄悄污染类型系统。  
> 先用 Schema 把它暴露出来，再用规则处理它。

# 14. 审计：找出有问题的记录

数据工程不是只看 `show()`。

我们需要系统性地检查：

- 哪些行的 `user_id` 是 null？
- 哪些行的 `duration_watched` 是 null？
- 哪些行可能需要修复？
- 哪些行必须丢弃？

现在我们先找出存在空值的记录。

In [ ]:
df_bronze.filter(
    F.col("user_id").isNull() | F.col("duration_watched").isNull()
).show(truncate=False)

+--------+-------+--------+----------------+-----------+-------------------+
|event_id|user_id|video_id|duration_watched|device_type|event_timestamp    |
+--------+-------+--------+----------------+-----------+-------------------+
|103     |NULL   |vid_c   |NULL            |iPhone     |2023-01-01T10:10:00|
|104     |502    |vid_a   |NULL            |iPad       |2023-01-01T10:15:00|
+--------+-------+--------+----------------+-----------+-------------------+



# 15. 统计脏数据数量

我们分别统计：

- 总行数
- `user_id` 为空的行数
- `duration_watched` 为空的行数

这一步在真实数据管道中非常重要。

因为你需要告诉团队：

> 今天有多少数据坏了？坏在哪里？

In [ ]:
total_rows = df_bronze.count()

bad_user_id_rows = df_bronze.filter(
    F.col("user_id").isNull()
).count()

missing_duration_rows = df_bronze.filter(
    F.col("duration_watched").isNull()
).count()

print("总行数:", total_rows)
print("user_id 为空的行数:", bad_user_id_rows)
print("duration_watched 为空的行数:", missing_duration_rows)

总行数: 6
user_id 为空的行数: 1
duration_watched 为空的行数: 2


## 小讨论

请思考：

1. `user_id` 为空的记录应该保留吗？
2. `duration_watched` 为空的记录应该删除，还是填成 0？
3. 这两个决策有什么不同？

---

## 推荐规则

在本实验中我们采用：

- 如果 `user_id` 为空：删除该行  
  因为没有用户 ID，这条播放事件无法可靠归属。
  
- 如果 `duration_watched` 为空：填充为 0  
  因为播放事件本身可能有效，只是观看时长缺失。

# 16. Silver 层：清洗数据

现在我们将 Bronze 数据清洗成 Silver 数据。

清洗规则：

1. 把 `event_timestamp` 字符串转换成真正的 timestamp
2. 把缺失的 `duration_watched` 填成 0
3. 删除没有 `user_id` 的行
4. 选择最终需要的字段
5. 保留 `event_id`，方便审计和追踪

---

## 注意

这一步不是简单地“删坏数据”。

我们是根据字段语义做不同处理：

- `user_id` 缺失：严重问题，删除
- `duration_watched` 缺失：可修复问题，填 0

In [ ]:
df_silver = (
    df_bronze
    .withColumn("timestamp", F.to_timestamp("event_timestamp"))
    .fillna(0, subset=["duration_watched"])
    .filter(F.col("user_id").isNotNull())
    .select(
        "event_id",
        "user_id",
        "video_id",
        "duration_watched",
        "device_type",
        "timestamp"
    )
)

# 17. 观察 Silver 数据

现在我们查看清洗后的数据。

In [ ]:
df_silver.show(truncate=False)

+--------+-------+--------+----------------+-----------+-------------------+
|event_id|user_id|video_id|duration_watched|device_type|timestamp          |
+--------+-------+--------+----------------+-----------+-------------------+
|101     |500    |vid_a   |120.5           |iPhone     |2023-01-01 10:00:00|
|102     |501    |vid_b   |30.0            |Android    |2023-01-01 10:05:00|
|104     |502    |vid_a   |0.0             |iPad       |2023-01-01 10:15:00|
|105     |503    |vid_d   |300.0           |Android    |2023-01-01 10:20:00|
|106     |504    |vid_e   |15.5            |iPhone     |2023-01-01 10:25:00|
+--------+-------+--------+----------------+-----------+-------------------+



# 18. 查看 Silver Schema

请观察：

- `timestamp` 是否已经变成 timestamp 类型
- `duration_watched` 是否仍然是 double
- `user_id` 是否仍然是 integer

In [ ]:
df_silver.printSchema()

root
 |-- event_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- video_id: string (nullable = true)
 |-- duration_watched: double (nullable = false)
 |-- device_type: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



## 关键教训

我们没有简单粗暴地删除所有“不完美”的数据。

原始第 104 行：

```text
104,502,vid_a,,iPad,2023-01-01T10:15:00
```

虽然缺失了观看时长，但 `user_id`、`video_id`、`device_type`、`timestamp` 都是有效的。

所以我们保留它，并把 `duration_watched` 填成 0。

这叫：

> 基于业务语义的数据清洗。

# 19. 验证清洗结果

清洗之后，我们不能只相信代码。

我们要验证几个条件：

1. Silver 表中不应该再有空的 `user_id`
2. `duration_watched` 不应该再有 null
3. 第 103 行应该被移除
4. 第 104 行应该被保留，并且 `duration_watched = 0`

In [ ]:
print("Silver 总行数:", df_silver.count())

print("Silver 中 user_id 为空的行数:",
      df_silver.filter(F.col("user_id").isNull()).count())

print("Silver 中 duration_watched 为空的行数:",
      df_silver.filter(F.col("duration_watched").isNull()).count())

Silver 总行数: 5
Silver 中 user_id 为空的行数: 0
Silver 中 duration_watched 为空的行数: 0


In [ ]:
print("检查 event_id = 103 是否还存在：")
df_silver.filter(F.col("event_id") == 103).show()

print("检查 event_id = 104 是否被保留：")
df_silver.filter(F.col("event_id") == 104).show()

检查 event_id = 103 是否还存在：
+--------+-------+--------+----------------+-----------+---------+
|event_id|user_id|video_id|duration_watched|device_type|timestamp|
+--------+-------+--------+----------------+-----------+---------+
+--------+-------+--------+----------------+-----------+---------+

检查 event_id = 104 是否被保留：
+--------+-------+--------+----------------+-----------+-------------------+
|event_id|user_id|video_id|duration_watched|device_type|          timestamp|
+--------+-------+--------+----------------+-----------+-------------------+
|     104|    502|   vid_a|             0.0|       iPad|2023-01-01 10:15:00|
+--------+-------+--------+----------------+-----------+-------------------+



## 课堂检查

请回答：

1. 为什么 `event_id = 103` 被删除？
2. 为什么 `event_id = 104` 被保留？
3. 这说明数据清洗不是简单的“有 null 就删”，而是什么？

# 20. Gold 层：生成面向分析的结果表

Silver 层是清洗后的明细事件。

Gold 层通常是面向分析师、报表或机器学习任务的高质量数据集。

现在我们创建一个简单的 Gold 表：

> 按设备类型统计播放事件数、总观看时长、平均观看时长。

这类表可以回答：

- 哪种设备产生的播放最多？
- 哪种设备总观看时间最高？
- 哪种设备平均观看更久？

In [ ]:
df_gold_device_stats = (
    df_silver
    .groupBy("device_type")
    .agg(
        F.count("*").alias("num_events"),
        F.sum("duration_watched").alias("total_duration"),
        F.avg("duration_watched").alias("avg_duration")
    )
    .orderBy("device_type")
)

# 21. 查看 Gold 表

注意：`groupBy` 是一个非常常见的数据工程操作。

它通常比 `select` / `filter` 更昂贵，因为它可能需要按照 key 重新组织数据。

In [ ]:
df_gold_device_stats.show(truncate=False)

+-----------+----------+--------------+------------+
|device_type|num_events|total_duration|avg_duration|
+-----------+----------+--------------+------------+
|Android    |2         |330.0         |165.0       |
|iPad       |1         |0.0           |0.0         |
|iPhone     |2         |136.0         |68.0        |
+-----------+----------+--------------+------------+



## Gold 表解释

这张表已经不再是原始事件日志。

它是面向业务问题的分析结果。

这就是为什么我们把它称为 Gold 层。

---

## 思考

如果产品经理问：

> iPhone 用户是不是比 Android 用户看得更久？

你不会直接让他们看原始 CSV。  
你会给他们一张像这样的 Gold 表。

# 22. 使用 `explain()` 查看 Silver 执行计划

在真实数据工程中，你不应该只问：

> 结果对不对？

还应该问：

> Spark 准备怎么执行？

`explain()` 可以展示 Spark 的执行计划。

本实验不要求你完全读懂每一行，但你应该开始认识几个词：

| 关键词 | 大致含义 |
|---|---|
| `FileScan` | 从文件中读取数据 |
| `Project` | 选择列或生成列 |
| `Filter` | 过滤行 |
| `HashAggregate` | 聚合计算 |
| `Exchange` | 数据重分布，通常意味着 Shuffle |

先看 Silver 层的计划。

In [ ]:
df_silver.explain()

== Physical Plan ==
*(1) Project [event_id#0, user_id#1, video_id#2, coalesce(nanvl(duration_watched#3, null), 0.0) AS duration_watched#90, device_type#4, cast(event_timestamp#5 as timestamp) AS timestamp#89]
+- *(1) Filter isnotnull(user_id#1)
   +- FileScan csv [event_id#0,user_id#1,video_id#2,duration_watched#3,device_type#4,event_timestamp#5] Batched: false, DataFilters: [isnotnull(user_id#1)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/admin/Documents/raw_logs.csv], PartitionFilters: [], PushedFilters: [IsNotNull(user_id)], ReadSchema: struct<event_id:int,user_id:int,video_id:string,duration_watched:double,device_type:string,event_...




# 23. 使用 `explain()` 查看 Gold 执行计划

现在看 Gold 聚合表的执行计划。

请留意是否出现：

- `HashAggregate`
- `Exchange`

如果出现 `Exchange`，通常意味着 Spark 需要在不同分区或机器之间重新分布数据。

这就是我们在理论课中提到的 Shuffle 直觉。

In [ ]:
df_gold_device_stats.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [device_type#4 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(device_type#4 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=410]
      +- HashAggregate(keys=[device_type#4], functions=[count(1), sum(duration_watched#90), avg(duration_watched#90)])
         +- Exchange hashpartitioning(device_type#4, 200), ENSURE_REQUIREMENTS, [plan_id=407]
            +- HashAggregate(keys=[device_type#4], functions=[partial_count(1), partial_sum(duration_watched#90), partial_avg(duration_watched#90)])
               +- Project [coalesce(nanvl(duration_watched#3, null), 0.0) AS duration_watched#90, device_type#4]
                  +- Filter isnotnull(user_id#1)
                     +- FileScan csv [user_id#1,duration_watched#3,device_type#4] Batched: false, DataFilters: [isnotnull(user_id#1)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/admin/Documents/raw_logs.csv], PartitionFilters: [], PushedFilters

## 观察问题

请你在输出中寻找：

1. 是否看到了 `FileScan`？
2. 是否看到了 `Project`？
3. 是否看到了 `Filter`？
4. 在 Gold 表中是否看到了 `HashAggregate`？
5. 是否看到了 `Exchange`？

---

## 关键教训

`select`、`withColumn`、`filter` 通常是逐行转换。

`groupBy` 这类操作可能需要重新组织数据，因此执行计划更复杂。

# 24. 为什么写成 Parquet？

CSV 适合交换数据，但不适合作为大规模分析存储格式。

Parquet 是大数据系统中非常常用的列式存储格式。

它的优势包括：

- 列式存储，适合只读取部分列
- 保存 Schema
- 通常压缩效果更好
- 被 Spark、Hive、Presto、Trino、DuckDB、BigQuery 等工具广泛支持

在数据湖中，Parquet 通常比 CSV 更适合作为清洗后数据的存储格式。

# 25. 为什么要分区？

假设下游经常问：

```sql
SELECT *
FROM silver_events
WHERE device_type = 'iPhone'
```

如果所有数据都在一个巨大文件中，查询引擎可能需要扫描大量无关数据。

如果我们按 `device_type` 分区，输出目录会变成：

```text
silver_events/
    device_type=iPhone/
    device_type=Android/
    device_type=iPad/
```

当查询只需要 iPhone 数据时，Spark 可以跳过其他目录。

组织结构。

这叫做：

> Partition Pruning，分区裁剪。

---

## 重要提醒

分区不是越多越好。

好的分区字段通常满足：

- 经常出现在过滤条件中
- 基数不要太高
- 每个分区数据量不要太小
- 能明显减少扫描范围

不要轻易按 `user_id` 分区，因为用户 ID 可能有几百万个，会制造大量小文件。

In [ ]:
# 将 df_silver 写出为按 device_type 分区的 Parquet 文件
silver_output_path = "silver_events"

df_silver.write \
    .mode("overwrite") \
    .partitionBy("device_type") \
    .parquet(silver_output_path)

# 26. 审计文件夹结构

数据工程师不能只相信一句：

```text
写入成功
```

你要检查磁盘上到底产生了什么。

现在我们遍历输出目录，看看 Spark 创建了哪些文件与文件夹。

In [ ]:
for root, dirs, files in os.walk(silver_output_path):
    level = root.replace(silver_output_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    
    subindent = "  " * (level + 1)
    for file in files:
        if not file.startswith("."):
            print(f"{subindent}{file}")

## 你应该看到类似结构

```text
silver_events/
  device_type=Android/
    part-....parquet
  device_type=iPad/
    part-....parquet
  device_type=iPhone/
    part-....parquet
```

这说明：

> 原本 DataFrame 里的 `device_type` 列，被转化成了物理目录结构。

这就是数据湖表组织方式的核心直觉之一。

# 27. 学生练习 A：数据质量审计

现在轮到你来扩展这个管道。

真实数据工程中，清洗数据之前，应该先做数据质量审计。

请你完成下面几个任务。

---

## 任务 A1

统计每一列的 null 数量。

目标输出类似：

| column_name | null_count |
|---|---|
| event_id | 0 |
| user_id | 1 |
| video_id | 0 |
| duration_watched | 2 |
| device_type | 0 |
| event_timestamp | 0 |

---

## 提示

你可以使用：

```python
F.col(c).isNull()
F.when(..., 1).otherwise(0)
F.sum(...)
```

也可以用循环逐列统计。

In [ ]:
# TODO: 练习 A1
# 统计 df_bronze 中每一列的 null 数量

null_counts = df_bronze.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_bronze.columns
])
null_counts.show()


# 28. 学生练习 B：增加新的清洗规则

现在我们假设业务团队给出了一条新规则：

> `duration_watched` 不应该为负数。

负数观看时长没有业务意义。

请你构造一份新的脏数据，其中包含一行负数观看时长，然后重新读取并清洗。

---

## 任务 B1

创建一个新的 CSV 文件：`raw_logs_v2.csv`

要求包含下面这一行：

```text
107,505,vid_f,-20.0,Android,2023-01-01T10:30:00
```

---

## 任务 B2

读取 `raw_logs_v2.csv`

仍然必须：

- 使用显式 Schema
- 使用 `PERMISSIVE`
- 不使用 `inferSchema`

---

## 任务 B3

构建新的 Silver 表，要求：

- 删除 `user_id` 为空的行
- `duration_watched` 缺失时填 0
- 删除 `duration_watched < 0` 的行
- 转换 timestamp

In [ ]:
# TODO: 练习 B1
# 创建 raw_logs_v2.csv，包含原始数据，并额外加入负数观看时长行

data_content_v2 = """event_id,user_id,video_id,duration_watched,device_type,event_timestamp
101,500,vid_a,120.5,iPhone,2023-01-01T10:00:00
102,501,vid_b,30.0,Android,2023-01-01T10:05:00
103,CORRUPT,vid_c,zero,iPhone,2023-01-01T10:10:00
104,502,vid_a,,iPad,2023-01-01T10:15:00
105,503,vid_d,300.0,Android,2023-01-01T10:20:00
106,504,vid_e,15.5,iPhone,2023-01-01T10:25:00
107,505,vid_f,-20.0,Android,2023-01-01T10:30:00
"""

with open("raw_logs_v2.csv", "w", encoding="utf-8") as f:
    f.write(data_content_v2)

print("包含负数观看时长的 raw_logs_v2.csv 已创建。")

# 练习 B2: 读取 raw_logs_v2.csv
df_bronze_v2 = spark.read \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .csv("raw_logs_v2.csv")

# 练习 B3: 构建新的 Silver 表
df_silver_v2 = (
    df_bronze_v2
    .withColumn("timestamp", F.to_timestamp("event_timestamp"))
    .fillna(0, subset=["duration_watched"])
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("duration_watched") >= 0)
    .select(
        "event_id",
        "user_id",
        "video_id",
        "duration_watched",
        "device_type",
        "timestamp"
    )
)
df_silver_v2.show(truncate=False)


# 29. 学生练习 C：构建新的 Gold 表

现在你已经有了清洗后的 `df_silver_v2`。

请你构建一个新的 Gold 表：

> 按 `video_id` 统计每个视频的播放事件数和总观看时长。

---

## 任务 C1

生成 `df_gold_video_stats`，包含字段：

- `video_id`
- `num_events`
- `total_duration`
- `avg_duration`

---

## 任务 C2

按照 `total_duration` 从高到低排序。

---

## 任务 C3

使用 `explain()` 查看执行计划。

请观察是否出现：

- `HashAggregate`
- `Exchange`

In [ ]:
# TODO: 练习 C
# 构建按 video_id 聚合的 Gold 表

df_gold_video_stats = (
    df_silver_v2
    .groupBy("video_id")
    .agg(
        F.count("*").alias("num_events"),
        F.sum("duration_watched").alias("total_duration"),
        F.avg("duration_watched").alias("avg_duration")
    )
    .orderBy(F.col("total_duration").desc())
)
df_gold_video_stats.show()
df_gold_video_stats.explain()


# 30. 关闭 SparkSession

在所有计算完成后，必须停止 SparkSession 以释放系统资源。

In [ ]:
spark.stop()